# Calibración GQ/H y MRDF

Este notebook calibra la curva backbone GQ/H contra `G/Gmax` y luego calibra MRDF contra amortiguamiento. `gamma_ref` ya no se toma del `gamma_50` de la curva objetivo; se calcula como `tau_max / Gmax`, consistente con GQ/H.

El ejemplo usa pocas iteraciones para ejecutarse rápido. Para producción, aumente `CalibrationSettings` o use los valores por defecto.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "dynaengine").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynaengine import CalibrationSettings, calibrate_dynamic_curve

## Curva objetivo

`strain` debe estar en deformación adimensional. El amortiguamiento puede ingresarse como fracción (`0.05`) o porcentaje (`5.0`); internamente se normaliza a porcentaje.

In [ ]:
USER_DEFINED_CURVE = {
    "strain": [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1],
    "ggmax": [0.995, 0.985, 0.955, 0.89, 0.72, 0.49, 0.25, 0.11, 0.04, 0.015, 0.006],
    "damp": [0.010, 0.011, 0.014, 0.023, 0.050, 0.090, 0.150, 0.190, 0.210, 0.205, 0.190],
}

GMAX_PA = 70_000_000
TAU_MAX_PA = 120_000
print(f"gamma_ref GQ/H = tau_max / Gmax = {TAU_MAX_PA / GMAX_PA:.6g}")

In [ ]:
settings = CalibrationSettings(
    gqh_init_points=2,
    gqh_iterations=3,
    mrdf_init_points=2,
    mrdf_iterations=3,
    random_state=1,
)

try:
    result = calibrate_dynamic_curve(
        USER_DEFINED_CURVE,
        gmax_pa=GMAX_PA,
        tau_max_pa=TAU_MAX_PA,
        settings=settings,
    )
    calibration_summary = {
        "gamma_ref": result.gamma_ref,
        "theta": result.theta,
        "mrdf": result.mrdf,
        "dmin": result.dmin,
        "gqh_score": result.gqh_score,
        "mrdf_score": result.mrdf_score,
    }
    calibration_summary
except ImportError as exc:
    print("Para ejecutar esta celda instale la dependencia opcional: bayesian-optimization")
    print(exc)

In [ ]:
try:
    comparison = pd.DataFrame(
        {
            "strain": result.strain,
            "target_ggmax": result.target_ggmax,
            "calibrated_ggmax": result.calibrated_ggmax,
            "target_damping_percent": result.target_damping_percent,
            "calibrated_damping_percent": result.calibrated_damping_percent,
        }
    )
    comparison.head()
except NameError:
    print("Ejecute primero la celda de calibración.")